# 🔀 Conditional Workflow — Challenge

## Scenario

You are building an automated **content pipeline** for a Casablanca tech blog.
The pipeline must:

1. **Write** a short article on a given topic (minimum 200 words)
2. **Review** the draft — approve it if it meets the word count, reject it otherwise
3. **Conditionally route** the workflow:
   - ✅ Approved → **Publish** (save as a Markdown file)
   - ❌ Rejected → Output a rejection message with the reason

---

## ⏱ Time Budget

- **Core task**: ~15 minutes
- **Optional task**: +5 minutes

---

## YOUR TASK (Core — ~15 min)

| Step | What you need to do |
|------|---------------------|
| 1 | Write **agent instructions** for all three agents (Writer, Reviewer, Publisher) |
| 2 | Define **Pydantic models** for the Writer and Reviewer output schemas |
| 3 | Implement the **`to_review_result` executor** that parses the reviewer's JSON output |
| 4 | Implement the **`select_targets` routing function** |
| 5 | Build the **workflow** using `WorkflowBuilder` with the correct edges |

---

## ⭐ OPTIONAL TASK (+5 min)

Add a **French Summary agent** as a 4th step that runs *after* publishing.
It should produce a 2-3 sentence summary of the article in French
(useful for Casablanca's bilingual business community).

You will need to:
- Create the agent and wrap it in an `AgentExecutor`
- Add an edge from `save_draft` → `french_summary_agent`
- Also add an edge from `french_summary_agent` → `publisher_agent`
  (so the publisher saves after the French summary is added)

---

> 💡 **Hints**:
> - Use `WorkflowBuilder.add_edge()` for linear steps and
>   `add_multi_selection_edge_group()` for the conditional branch.
> - The `@executor` decorator marks functions that the workflow engine calls.
> - `ctx.send_message()` passes data to the next step;
>   `ctx.yield_output()` returns a final output to the caller.

In [ ]:
####### Linux / MacOS Setup Instructions #######
# ! sudo apt update && sudo apt install graphviz -y   # Linux
# ! brew install graphviz                              # macOS

In [ ]:
# ! pip install -r ../../../Installation/requirements.txt -U

In [ ]:
import os
from typing import cast
from dataclasses import dataclass
from typing_extensions import Literal
from pydantic import BaseModel

In [ ]:
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider
from agent_framework import (
    AgentExecutor,
    AgentResponse,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    executor,
    WorkflowViz
)

In [ ]:
load_dotenv()

In [ ]:
# ============================================================
# YOUR TASK 1: Write Agent Instructions
# ============================================================
# Define instructions for three agents:
#
# WriterInstructions:
#   - Write a 200+ word article on the given topic
#   - Output as JSON: { "draft_content": "<article text>" }
#   - The article should be well-structured with a title and paragraphs
#
# ReviewerInstructions:
#   - Check if the draft is 200+ words
#   - Output as JSON: { "review_result": "Yes" or "No",
#                       "reason": "<explanation>",
#                       "draft_content": "<original draft>" }
#
# PublisherInstructions:
#   - Save the draft content as a Markdown file
#   - Use the current timestamp as filename (e.g. 20260320143022.md)
#   - Use the code interpreter tool to write the file
#
# TODO: Fill in all three instruction strings below
# ============================================================

WriterInstructions = """TODO: Write instructions for the content writer agent"""

ReviewerInstructions = """TODO: Write instructions for the content reviewer agent"""

PublisherInstructions = """TODO: Write instructions for the publisher agent"""

In [ ]:
# Topic for the Casablanca tech blog article
ARTICLE_TOPIC = """
# Article Topic: The Rise of AI in Casablanca

Write about how artificial intelligence is transforming businesses in Casablanca,
Morocco. Cover at least these three aspects:
1. The growth of the tech startup scene in Casablanca
2. How local enterprises are adopting AI tools
3. Morocco's government digital transformation initiatives

Note: Do NOT include any code examples — prose only.
"""

In [ ]:
# ============================================================
# YOUR TASK 2: Define Pydantic Output Models
# ============================================================
# Define two Pydantic models that match the JSON output schemas
# described in your agent instructions above:
#
# WriterOutput: should have a `draft_content` field (str)
# ReviewerOutput: should have:
#   - review_result: Literal["Yes", "No"]
#   - reason: str
#   - draft_content: str
#
# Also define a ReviewResult dataclass (for passing between executors):
#   - review_result: str
#   - reason: str
#   - draft_content: str
#
# TODO: Define the three classes below
# ============================================================

# TODO: class WriterOutput(BaseModel): ...

# TODO: class ReviewerOutput(BaseModel): ...

# TODO: @dataclass class ReviewResult: ...

In [ ]:
# ============================================================
# YOUR TASK 3: Implement the Reviewer Executor
# ============================================================
# This executor receives the reviewer agent's raw JSON response,
# parses it using ReviewerOutput, and sends a ReviewResult to the
# next step via ctx.send_message().
#
# TODO: Complete the function body
# ============================================================

@executor(id="to_review_result")
async def to_review_result(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    response_text = response.agent_response.text.strip()
    print(f"Reviewer raw response: {response_text}")

    # TODO: Parse response_text using ReviewerOutput.model_validate_json()
    # TODO: Build a ReviewResult and call ctx.send_message() with it
    pass


# ============================================================
# YOUR TASK 4: Implement the Routing Function
# ============================================================
# select_targets receives the ReviewResult and a list of two target IDs:
#   target_ids[0] = handle_rejection executor
#   target_ids[1] = save_draft executor
#
# Return [target_ids[1]] (save_draft) if review_result == "Yes"
# Return [target_ids[0]] (handle_rejection) otherwise
#
# TODO: Complete the function body
# ============================================================

def select_targets(review: "ReviewResult", target_ids: list[str]) -> list[str]:
    handle_rejection_id, save_draft_id = target_ids
    # TODO: implement conditional routing
    pass


@executor(id="handle_rejection")
async def handle_rejection(review: "ReviewResult", ctx: WorkflowContext) -> None:
    """Outputs a rejection message when the draft does not pass review."""
    await ctx.yield_output(f"Draft rejected: {review.reason}. Please revise and resubmit.")


@executor(id="save_draft")
async def save_draft(review: "ReviewResult", ctx: WorkflowContext) -> None:
    """Sends the approved draft to the publisher agent."""
    await ctx.send_message(
        AgentExecutorRequest(
            messages=[Message("user", text=review.draft_content)],
            should_respond=True
        )
    )

In [ ]:
from IPython.display import SVG, display, HTML

class BlogWorkflowEvent(WorkflowEvent): ...

In [ ]:
async with (
    AzureCliCredential() as credential,
    AzureAIAgentsProvider(credential=credential) as provider,
):
    writer_agent_obj = None
    reviewer_agent_obj = None
    publisher_agent_obj = None
    try:
        client = AzureAIAgentClient(credential=credential)
        code_interpreter_tool = client.get_code_interpreter_tool()

        writer_agent_obj = await provider.create_agent(
            name="writer-agent",
            instructions=WriterInstructions,
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        )
        writer_agent = AgentExecutor(writer_agent_obj, id="writer_agent")

        reviewer_agent_obj = await provider.create_agent(
            name="reviewer-agent",
            instructions=ReviewerInstructions,
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        )
        reviewer_agent = AgentExecutor(reviewer_agent_obj, id="reviewer_agent")

        publisher_agent_obj = await provider.create_agent(
            name="publisher-agent",
            instructions=PublisherInstructions,
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
            tools=[code_interpreter_tool],
        )
        publisher_agent = AgentExecutor(publisher_agent_obj, id="publisher_agent")

        # ============================================================
        # YOUR TASK 5: Build the Workflow
        # ============================================================
        # Chain the agents using WorkflowBuilder:
        #   writer_agent
        #       → reviewer_agent
        #       → to_review_result        (executor)
        #       → [handle_rejection  OR  save_draft]  ← conditional branch
        #       → publisher_agent         (only from save_draft path)
        #
        # Use:
        #   .add_edge(source, target)
        #   .add_multi_selection_edge_group(
        #       source=to_review_result,
        #       targets=[handle_rejection, save_draft],
        #       selection_func=select_targets
        #   )
        #
        # TODO: Build and assign `workflow` below
        # ============================================================

        workflow = None  # TODO: replace with WorkflowBuilder(...).build()

        # Visualise the workflow (provided)
        print("Workflow visualization:")
        viz = WorkflowViz(workflow)
        print(viz.to_mermaid())
        svg_file = viz.export(format="svg")
        if svg_file and os.path.exists(svg_file):
            try:
                display(SVG(filename=svg_file))
            except Exception:
                pass

        # Run the workflow
        task = (
            "Write a blog article for the Casablanca Tech Blog based on the "
            "following topic. Once written, the reviewer will check it. If approved, "
            "the publisher will save it as a Markdown file.\n\n" + ARTICLE_TOPIC
        )

        events = await workflow.run(task)

        outputs = events.get_outputs()
        outputs = cast(list[AgentResponse], outputs)
        for output in outputs:
            print(f"{output.messages[0].author_name}: {output.text}\n")

        print("Final state:", events.get_final_state())

    finally:
        for agent_obj in [writer_agent_obj, reviewer_agent_obj, publisher_agent_obj]:
            if agent_obj is not None:
                try:
                    await provider._agents_client.delete_agent(agent_obj.id)
                except Exception:
                    pass
        print("done")

---

## ⭐ OPTIONAL TASK: Add a French Summary Agent

After the publisher saves the file, add a **4th agent** that creates a short
French-language summary (2-3 sentences) of the approved article — useful for
Casablanca's bilingual business audience.

**Steps:**
1. Define `FrenchSummaryInstructions` — the agent should read the draft and output
   a 2-3 sentence summary in French.
2. Create the agent with `provider.create_agent()` and wrap in `AgentExecutor`.
3. Add an edge: `save_draft → french_summary_agent`.
4. Change the existing edge from `save_draft → publisher_agent`
   to `french_summary_agent → publisher_agent`.

Rebuild the workflow and run it to see the French summary appear in the output.